# Notebook 10 — Testing & debugging mindset

| | |
|---|---|
| **Track** | AI Engineering · `03-python-foundations/exercises/` |
| **Level** | Advanced |
| **Pattern** | *Concept* → *Runnable demo* → *Your turn* → *Solution* |

**Prerequisites:** `09-async-io.ipynb`

**Next up:** `11-collections-itertools-functools.ipynb` (extended track); then `../python-foundations-beginner-to-advanced.ipynb`

---

## Learning objectives

- Write assertions for pure helpers.
- Isolate filesystem tests with tmp paths.
- Think about pytest adoption next.

## Table of contents

1. **Assertions & param tables**
2. **Tmp paths**
3. **Debugging hooks**
4. **Progressive drills — expected failures → float tolerances → canonical dict hashes**
5. **Exercise — test `truncate`**

---

## How to use this notebook

1. Run cells **top to bottom** the first time through.
2. Re-run and **change values** to test your understanding.
3. Do **Your turn** cells before opening solutions (HTML `<details>` blocks or later cells).

---


## 1 · Assertions

*Explanation:* Notebook-friendly checks — migrate to `pytest` later.


In [ ]:
def add(a: int, b: int) -> int:
    return a + b

assert add(2, 3) == 5
print("assertions pass")


## 2 · Table-driven checks

*Explanation:* Loop over cases — preview of parametrization.


In [ ]:
cases = [
    ("{}", {}),
    ('{"a":1}', {"a": 1}),
]
import json
for raw, expected in cases:
    assert json.loads(raw) == expected
print("table OK")


---

## Progressive drills — **easy → harder**

Testing mindset prevents regressions when prompts/tools change hourly—practice **negative paths**, **float fuzz**, **snapshot comparisons**.


### A · Easiest — expect failures (`pytest` preview)

Guard rails should explode loudly when inputs violate contracts.


In [ ]:
def budget_tokens(tokens: int) -> None:
    if tokens <= 0:
        raise ValueError("tokens must be positive")


try:
    budget_tokens(-5)
except ValueError:
    print("caught invalid budget")


### B · Medium — floating comparisons

Never assert raw `==` on uninformed embeddings analog math.


In [ ]:
def approx_eq(a: float, b: float, eps: float = 1e-9) -> bool:
    return abs(a - b) <= eps


assert approx_eq(0.1 + 0.2, 0.3)
print("float guard ok")


### C · Harder — canonical JSON fingerprints

Stable dumps detect accidental dict reorder regressions.


In [ ]:
import json


def canonical_json(payload: dict) -> str:
    return json.dumps(payload, sort_keys=True)


left = {"model": "mini", "kwargs": {"top_k": 8}}
right = {"kwargs": {"top_k": 8}, "model": "mini"}
assert canonical_json(left) == canonical_json(right)
print("snapshots match")


### Exercise — test `truncate`

Define `truncate(s, n)` returning `s` if `len(s) <= n` else `s[:n] + "…"`. Add **three** `assert` lines: one for a string shorter than `n`, one equal to `n`, and one longer than `n`.


In [ ]:
def truncate(s: str, n: int) -> str:
    raise NotImplementedError


# Your asserts here


### Solution — truncate

<details>
<summary>Click to expand</summary>

```python
def truncate(s: str, n: int) -> str:
    if len(s) <= n:
        return s
    return s[:n] + "…"

assert truncate("hi", 5) == "hi"
assert truncate("abcd", 4) == "abcd"
assert truncate("abcdefghij", 4) == "abcd…"
```

</details>
